# Writer Palmyra X4 and X5 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Writer's Palmyra X5 and X4 are text models for enterprise writing and tool use. X5 is
the newer one and the default here. Unlike Palmyra Vision
([`01-palmyra-vision`](01-palmyra-vision.ipynb)), both accept a system prompt and tool
definitions, so check each Writer model's limits on its own.

| Model | ID to call on `bedrock-runtime` | Input | Notes |
|---|---|---|---|
| Palmyra X5 | `us.writer.palmyra-x5-v1:0` | Text | The default here |
| Palmyra X4 | `us.writer.palmyra-x4-v1:0` | Text | Previous generation |

IDs as of September 2026, in `us-east-1`. These models are on `bedrock-runtime` only, so
every call here uses **Converse**. Both accept only an inference profile, such as the
`us.` profiles above.

**Covered:** first call, streaming, multi-turn with a system prompt, tool use, and
structured output through a tool.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). Every call here goes through boto3, which signs each request with your AWS
credentials, so there is no API key to mint. The setup cell checks that each inference
profile exists in your Region. Converse and inference profiles are explained in
[`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb).

In [1]:
import json

import boto3

REGION = "us-east-1"
X5 = "us.writer.palmyra-x5-v1:0"
X4 = "us.writer.palmyra-x4-v1:0"

runtime = boto3.client("bedrock-runtime", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)

profiles = {
    p["inferenceProfileId"]
    for page in bedrock.get_paginator("list_inference_profiles").paginate()
    for p in page["inferenceProfileSummaries"]
}
for model in (X5, X4,):
    print(f"{model:27} {'available' if model in profiles else 'no such profile in ' + REGION}")


def text_of(response: dict) -> str:
    """The text blocks of a Converse reply, joined. Never assume content[0] is text."""
    return "".join(b["text"] for b in response["output"]["message"]["content"] if "text" in b)

us.writer.palmyra-x5-v1:0   available
us.writer.palmyra-x4-v1:0   available


## 1. First call

`content` is a list of blocks, and settings such as the token limit go in
`inferenceConfig`. The reply's `stopReason` says why it ended: `end_turn` is a complete
answer, and `max_tokens` means the budget ran out first.

In [2]:
response = runtime.converse(
    modelId=X5,
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 500},
)
print(text_of(response).strip())
print(f"stopReason={response['stopReason']}  totalTokens={response['usage']['totalTokens']}")

Canberra
stopReason=end_turn  totalTokens=251


## 2. Streaming

`converse_stream` returns an event stream. Text arrives in `contentBlockDelta` events,
and `messageStop` carries the stop reason.

In [3]:
response = runtime.converse_stream(
    modelId=X5,
    messages=[{"role": "user", "content": [{"text": "Count from 1 to 10, separated by commas."}]}],
    inferenceConfig={"maxTokens": 500},
)
for event in response["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"].get("text", ""), end="", flush=True)
    elif "messageStop" in event:
        print(f"\nstopReason={event['messageStop']['stopReason']}")

1

,

2

,

3

...

9

,

1

0


stopReason=end_turn


## 3. Multi-turn and a system prompt

Converse keeps no state between calls, so each request carries the whole conversation.
Append the model's reply as it arrived, then the next user turn. A system prompt goes in
the separate `system` parameter.

In [4]:
messages = [{"role": "user", "content": [{"text": "My favourite colour is teal. Reply with one word: noted."}]}]
system = [{"text": "You are a concise assistant."}]

first = runtime.converse(modelId=X5, messages=messages, system=system, inferenceConfig={"maxTokens": 500})
messages.append(first["output"]["message"])
messages.append({"role": "user", "content": [{"text": "What is my favourite colour? One word."}]})
second = runtime.converse(modelId=X5, messages=messages, system=system, inferenceConfig={"maxTokens": 500})

print("turn 1:", text_of(first).strip())
print("turn 2:", text_of(second).strip())

turn 1: noted
turn 2: teal


## 4. Tool use

A Converse tool is a `toolSpec` whose JSON Schema sits under `inputSchema.json`. The
loop sends the tools, runs each `toolUse` the model returns, answers each with a
`toolResult` carrying the same `toolUseId`, and repeats until `stopReason` is no longer
`tool_use`. Check each call's input before running it: `stopReason: tool_use` means the
model wants a tool, and says nothing about whether the input is usable.

In [5]:
def lookup_inventory(sku: str) -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0}
    if sku.upper() not in stock:
        return {"sku": sku, "error": "unknown SKU"}
    return {"sku": sku, "quantity": stock[sku.upper()]}


TOOLS = [{
    "toolSpec": {
        "name": "lookup_inventory",
        "description": "Current stock level for a SKU.",
        "inputSchema": {"json": {
            "type": "object",
            "properties": {"sku": {"type": "string", "description": "SKU code, e.g. A-100"}},
            "required": ["sku"],
        }},
    },
}]
IMPLEMENTATIONS = {"lookup_inventory": lookup_inventory}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's schema."""
    spec = next((t["toolSpec"] for t in TOOLS if t["toolSpec"]["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    schema = spec["inputSchema"]["json"]
    missing = [k for k in schema["required"] if k not in args]
    unknown = [k for k in args if k not in schema["properties"]]
    wrong_type = [k for k in args if k in schema["properties"] and schema["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return IMPLEMENTATIONS[name](**args)

In [6]:
messages = [{"role": "user", "content": [{"text": "How many units of SKU A-100 do we have?"}]}]
for _ in range(5):  # bounded, so a model that keeps calling tools cannot loop forever
    response = runtime.converse(
        modelId=X5, messages=messages, toolConfig={"tools": TOOLS},
        inferenceConfig={"maxTokens": 1000},
    )
    message = response["output"]["message"]
    messages.append(message)
    uses = [b["toolUse"] for b in message["content"] if "toolUse" in b]
    if response["stopReason"] != "tool_use" or not uses:
        break
    results = []
    for use in uses:
        result = run_tool(use["name"], use["input"])
        print(f"tool call: {use['name']}({use['input']}) -> {result}")
        results.append({"toolResult": {"toolUseId": use["toolUseId"], "content": [{"json": result}]}})
    messages.append({"role": "user", "content": results})
else:
    raise RuntimeError("no answer after 5 rounds of tool calls")

print("answer:", text_of(response).strip())

tool call: lookup_inventory({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42}


answer: We currently have 42 units of SKU A-100 in stock. Let me know if you need help with anything else!


## 5. Structured output

Converse has no `response_format`. The usual way to get JSON back is a tool whose schema
is the shape you want: the model returns the fields as the tool's `input`, already
parsed. Palmyra X refuses a forced choice: `toolChoice` set to `tool` or `any` returns
`ValidationException: This model doesn't support the toolConfig.toolChoice.tool field.
Remove toolConfig.toolChoice.tool and try again.` So the request leaves the choice on
`auto`, and the cell stops if no tool call comes back. A schema controls the shape and
has no say over the values, so the cell checks each value against the source text.

In [7]:
RECORD = {
    "toolSpec": {
        "name": "record_invoice",
        "description": "Record the fields of an invoice.",
        "inputSchema": {"json": {
            "type": "object",
            "properties": {
                "invoice_id": {"type": "string"},
                "vendor": {"type": "string"},
                "total": {"type": "number"},
                "currency": {"type": "string"},
            },
            "required": ["invoice_id", "vendor", "total", "currency"],
        }},
    },
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}
PROMPT = ("Record this invoice. Write the total as a number without a thousands separator, "
          f"so 2,345.60 becomes 2345.6.\n\n{SOURCE}")

response = runtime.converse(
    modelId=X5,
    messages=[{"role": "user", "content": [{"text": PROMPT}]}],
    toolConfig={"tools": [RECORD], "toolChoice": {"auto": {}}},
    inferenceConfig={"maxTokens": 1000},
)
calls = [b["toolUse"]["input"] for b in response["output"]["message"]["content"] if "toolUse" in b]
if not calls:
    raise RuntimeError(f"no tool call came back (stopReason={response['stopReason']})")

invoice = calls[0]
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in EXPECTED.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the source


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Bare model ID | `ValidationException: Invocation of model ID … with on-demand throughput isn’t supported` | Call the inference profile ID, such as `us.<model-id>`. |
| Reading the answer | `content[0]` is not always the text | Read content blocks by type (`text`, `toolUse`, `reasoningContent`). |
| Structured output values | A tool call whose input holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | `ThrottlingException` under load | Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Forced tool choice | `ValidationException: This model doesn't support the toolConfig.toolChoice.tool field.` | Leave `toolChoice` on `auto` and check that a call came back (section 5). |
| Images | `ValidationException: This model doesn't support the image content block that you provided.` | Palmyra X is text only. Send images to Palmyra Vision. |

## Next

- Writer's vision model, on both endpoints:
  [`01-palmyra-vision`](01-palmyra-vision.ipynb)
- Other families on Converse only:
  [`13-amazon-nova/01`](../13-amazon-nova/01-nova-ladder-and-vision.ipynb),
  [`15-meta-llama/01`](../15-meta-llama/01-llama4-moe-and-vision.ipynb)
- Converse and inference profiles in depth:
  [`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)